# ChefAI - Generateur de Recettes avec IA Generative
**Projet individuel | ENSTAB 2026**
Modele : LLaMA 3.3 70B via Groq (LLM - Gratuit)
---

In [ ]:
# CELLULE 1 : Installation
!pip install groq gradio -q
print('Installation terminee !')

In [ ]:
# CELLULE 2 : Imports
from groq import Groq
import gradio as gr
import json
import re
print('Imports OK')

In [ ]:
# CELLULE 3 : Cle API Groq (gratuite sur https://console.groq.com)
API_KEY = 'gsk_XXXXXXXXXXXXXXXXXXXX'   # <-- colle ta cle Groq ici
client = Groq(api_key=API_KEY)
print('Cle API Groq configuree !')

In [ ]:
# CELLULE 4 : Fonction de generation de recette (LLM LLaMA via Groq)

def generer_recette(ingredients_texte, cuisine, temps, personnes):
    if not ingredients_texte.strip():
        return "Veuillez entrer au moins un ingredient."

    cuisine_str = cuisine if cuisine != "Toutes" else "au choix"

    prompt = (
        "Tu es un chef cuisinier expert et creatif.\n\n"
        "Ingredients disponibles : " + ingredients_texte + "\n"
        "Cuisine souhaitee : " + cuisine_str + "\n"
        "Temps disponible : " + temps + "\n"
        "Nombre de personnes : " + str(personnes) + "\n\n"
        "Genere une recette complete et delicieuse.\n"
        "Reponds UNIQUEMENT en JSON valide (sans markdown, sans backticks) :\n"
        "{\n"
        "  \"titre\": \"Nom creatif de la recette\",\n"
        "  \"temps_preparation\": \"X minutes\",\n"
        "  \"temps_cuisson\": \"X minutes\",\n"
        "  \"difficulte\": \"Facile / Moyen / Difficile\",\n"
        "  \"calories\": \"environ XXX kcal par portion\",\n"
        "  \"ingredients\": [\"quantite + ingredient 1\", \"quantite + ingredient 2\"],\n"
        "  \"etapes\": [\"Etape 1 detaillee\", \"Etape 2 detaillee\"],\n"
        "  \"conseil_chef\": \"Un conseil pour reussir la recette\",\n"
        "  \"variante\": \"Une variante possible\"\n"
        "}"
    )

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )
        reponse = response.choices[0].message.content
        reponse = re.sub(r'```json|```', '', reponse).strip()
        recette = json.loads(reponse)

        resultat  = "# " + recette['titre'] + "\n\n---\n\n"
        resultat += "### Informations\n| | |\n|---|---|\n"
        resultat += "| Preparation | " + recette['temps_preparation'] + " |\n"
        resultat += "| Cuisson     | " + recette['temps_cuisson'] + " |\n"
        resultat += "| Difficulte  | " + recette['difficulte'] + " |\n"
        resultat += "| Calories    | " + recette['calories'] + " |\n"
        resultat += "| Personnes   | " + str(personnes) + " |\n\n---\n\n"
        resultat += "### Ingredients\n"
        for ing in recette['ingredients']:
            resultat += "- " + ing + "\n"
        resultat += "\n---\n\n### Preparation\n"
        for i, etape in enumerate(recette['etapes'], 1):
            resultat += "**Etape " + str(i) + "** - " + etape + "\n\n"
        resultat += "---\n\n### Conseil du Chef\n> " + recette['conseil_chef'] + "\n"
        resultat += "\n### Variante\n> " + recette['variante'] + "\n"
        return resultat

    except json.JSONDecodeError:
        return "Erreur JSON. Reponse brute :\n\n" + reponse
    except Exception as e:
        return "Erreur : " + str(e)

# TEST : voir le resultat directement
print("Test du modele en cours...")
resultat_test = generer_recette("poulet, tomates, ail", "Tunisienne", "Moyen (30 a 45 min)", 4)
print(resultat_test)
print("Fonction prete !")


In [ ]:
# CELLULE 5 : Interface Web Gradio + Lancement

def construire_interface():
    with gr.Blocks(
        title="ChefAI - Generateur de Recettes",
        theme=gr.themes.Soft(primary_hue="orange", secondary_hue="green"),
        css="footer { display: none !important; }"
    ) as demo:

        gr.Markdown("""
# ChefAI - Generateur de Recettes
**Entrez vos ingredients, l'IA genere une recette sur mesure !**
---""")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### Parametres")

                ingredients = gr.Textbox(
                    label="Ingredients disponibles",
                    placeholder="ex : poulet, tomates, ail, oignons...",
                    lines=4,
                    info="Separez les ingredients par des virgules"
                )

                gr.Markdown("**Suggestions rapides :**")
                with gr.Row():
                    gr.Button("Poulet").click(lambda: "poulet, tomates, ail, oignons", outputs=ingredients)
                    gr.Button("Poisson").click(lambda: "poisson, citron, ail, persil", outputs=ingredients)
                with gr.Row():
                    gr.Button("Legumes").click(lambda: "carottes, pommes de terre, oignons, ail", outputs=ingredients)
                    gr.Button("Pates").click(lambda: "pates, tomates, ail, basilic, fromage", outputs=ingredients)

                cuisine = gr.Dropdown(
                    label="Type de cuisine",
                    choices=["Toutes", "Tunisienne", "Francaise", "Italienne", "Orientale", "Mediterraneenne", "Asiatique"],
                    value="Tunisienne"
                )

                temps = gr.Radio(
                    label="Temps disponible",
                    choices=["Rapide (moins de 20 min)", "Moyen (30 a 45 min)", "Long (plus d une heure)"],
                    value="Moyen (30 a 45 min)"
                )

                personnes = gr.Slider(label="Nombre de personnes", minimum=1, maximum=10, value=4, step=1)
                btn = gr.Button("Generer ma recette", variant="primary", size="lg")

            with gr.Column(scale=2):
                gr.Markdown("### Recette generee")
                output = gr.Markdown(value="*La recette apparaitra ici apres generation...*")

        btn.click(
            fn=generer_recette,
            inputs=[ingredients, cuisine, temps, personnes],
            outputs=output,
            show_progress=True
        )

        gr.Markdown("---\nProjet IA Generative - ENSTAB 2026 | Modele : LLaMA 3.3 70B via Groq (LLM)")

    return demo

demo = construire_interface()
demo.launch(share=True, debug=True)
# Le lien gradio.live qui s affiche = ton interface web !
# Copie-le pour ta demo video et ton README GitHub !
